<a href="https://colab.research.google.com/github/Shizukem/cu-i-k-/blob/main/b%C3%A0i_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
P_S = {'s1': 0.30, 's2': 0.45, 's3': 0.20, 's4': 0.05}

# β_j: hệ số hiệu quả first-stage
BETA = {'I': 1.00, 'D': 1.10, 'AI': 1.25, 'H': 0.95}

# β^s_j: hệ số hiệu quả second-stage theo kịch bản
BETA_S = {
    ('s1','I'):1.25, ('s1','D'):1.35, ('s1','AI'):1.55, ('s1','H'):1.05,
    ('s2','I'):1.00, ('s2','D'):1.10, ('s2','AI'):1.25, ('s2','H'):0.95,
    ('s3','I'):0.75, ('s3','D'):0.85, ('s3','AI'):0.90, ('s3','H'):1.00,
    ('s4','I'):0.40, ('s4','D'):0.50, ('s4','AI'):0.55, ('s4','H'):1.10,
}

BUDGET1  = 65_000   # tỷ VND — ngân sách first-stage (giữ lại 15.000 dự phòng)
RESERVE  = 15_000   # tỷ VND — ngân sách second-stage dự phòng
BUDGET2  = {'s1': 15_000, 's2': 15_000, 's3': 12_000, 's4': 8_000}
MIN_H    = 15_000   # sàn nhân lực số first-stage
MIN_ITEM =  3_000   # sàn tối thiểu từng hạng mục

# Thông tin kịch bản kinh tế
SCEN_INFO = {
    's1': {'label':'Lạc quan',    'gdp_world':3.5, 'fdi':32.0, 'export':12.0},
    's2': {'label':'Cơ sở',       'gdp_world':2.8, 'fdi':27.0, 'export': 8.0},
    's3': {'label':'Bi quan',     'gdp_world':1.5, 'fdi':20.0, 'export': 3.0},
    's4': {'label':'Khủng hoảng', 'gdp_world':0.2, 'fdi':12.0, 'export':-5.0},
}


In [ ]:
def build_SP(fix_x=None, beta_s_override=None,
             budget2_override=None, scenarios=None):
    """
    Xây dựng và giải bài toán Two-Stage SP bằng Pyomo + HiGHS.

    fix_x           : dict {j: val} — cố định x (cho EEV)
    beta_s_override : dict {(s,j): val} — override beta_s (cho EV)
    budget2_override: dict {s: val}     — override budget2 (cho EV)
    scenarios       : list — tập kịch bản dùng (mặc định SCENS)
    """
    sc  = scenarios if scenarios else SCENS
    b2  = budget2_override if budget2_override else BUDGET2
    bs  = beta_s_override if beta_s_override else BETA_S

    m = pyo.ConcreteModel()
    m.J = pyo.Set(initialize=ITEMS)
    m.S = pyo.Set(initialize=sc)

    m.x = pyo.Var(m.J, within=pyo.NonNegativeReals)
    m.y = pyo.Var(m.S, m.J, within=pyo.NonNegativeReals)

    # Cố định x nếu đánh giá EEV
    if fix_x:
        for j in ITEMS:
            m.x[j].fix(fix_x[j])

    # ── Hàm mục tiêu ────────────────────────────────────────
    p_use = {s: P_S[s] for s in sc}
    total_p = sum(p_use.values())
    # Chuẩn hóa xác suất nếu chỉ dùng 1 kịch bản
    if len(sc) == 1:
        p_use = {sc[0]: 1.0}

    m.obj = pyo.Objective(
        expr=(sum(BETA[j] * m.x[j] for j in ITEMS)
            + sum(p_use[s] * bs[s, j] * m.y[s, j]
                  for s in sc for j in ITEMS)),
        sense=pyo.maximize,
    )

    # ── Ràng buộc ────────────────────────────────────────────
    # C1: Ngân sách first-stage
    m.c1 = pyo.Constraint(
        expr=sum(m.x[j] for j in ITEMS) <= BUDGET1)

    # C4: Sàn nhân lực số (H là "hàng hóa bảo hiểm")
    m.c4 = pyo.Constraint(expr=m.x['H'] >= MIN_H)

    # C5: Sàn tối thiểu từng hạng mục
    def c5_rule(m, j):
        return m.x[j] >= MIN_ITEM
    m.c5 = pyo.Constraint(m.J, rule=c5_rule)

    # C2: Ngân sách second-stage theo kịch bản
    def c2_rule(m, s):
        return sum(m.y[s, j] for j in ITEMS) <= b2[s]
    m.c2 = pyo.Constraint(m.S, rule=c2_rule)

    # C3: Mở rộng AI bị giới hạn bởi nhân lực đã đào tạo
    def c3_rule(m, s):
        return m.y[s, 'AI'] <= 0.5 * m.x['H']
    m.c3 = pyo.Constraint(m.S, rule=c3_rule)

    # ── Giải ────────────────────────────────────────────────
    solver = SolverFactory('highs')
    result = solver.solve(m, tee=False)

    if str(result.solver.status) != 'ok':
        return {'feasible': False}

    x_val = {j: pyo.value(m.x[j]) for j in ITEMS}
    y_val = {(s, j): pyo.value(m.y[s, j])
             for s in sc for j in ITEMS}
    Z_val = pyo.value(m.obj)

    # Shadow prices
    sp = {}
    try:
        sp['C1_budget'] = pyo.value(m.dual[m.c1]) if hasattr(m, 'dual') else 0
    except Exception:
        sp = {}

    return {
        'feasible': True,
        'Z': Z_val,
        'x': x_val,
        'y': y_val,
        'model': m,
    }


def build_WS(s0):
    """
    Wait-and-See: biết trước kịch bản s0 → tối ưu x + y^s0 cùng lúc.
    """
    m = pyo.ConcreteModel()
    m.J = pyo.Set(initialize=ITEMS)
    m.x = pyo.Var(m.J, within=pyo.NonNegativeReals)
    m.y = pyo.Var(m.J, within=pyo.NonNegativeReals)

    m.obj = pyo.Objective(
        expr=(sum(BETA[j]     * m.x[j] for j in ITEMS)
            + sum(BETA_S[s0, j] * m.y[j] for j in ITEMS)),
        sense=pyo.maximize,
    )
    m.c1  = pyo.Constraint(expr=sum(m.x[j] for j in ITEMS) <= BUDGET1)
    m.c4  = pyo.Constraint(expr=m.x['H'] >= MIN_H)
    def c5_ws(m, j): return m.x[j] >= MIN_ITEM
    m.c5  = pyo.Constraint(m.J, rule=c5_ws)
    m.c2  = pyo.Constraint(expr=sum(m.y[j] for j in ITEMS) <= BUDGET2[s0])
    m.c3  = pyo.Constraint(expr=m.y['AI'] <= 0.5 * m.x['H'])

    SolverFactory('highs').solve(m, tee=False)

    return {
        'Z': pyo.value(m.obj),
        'x': {j: pyo.value(m.x[j]) for j in ITEMS},
        'y': {j: pyo.value(m.y[j]) for j in ITEMS},
    }


def build_robust_minimax():
    """
    Câu 10.5.4: Robust optimization — minimax regret.
    Tối thiểu hóa regret tối đa trên mọi kịch bản.
    Regret(x, s) = Z_WS(s) - [β·x + β^s·y^s(x)]
    min_x  max_s Regret(x,s)  ↔  min_x  θ  s.t. θ ≥ Regret(x,s) ∀s
    """
    # Lấy Z_WS(s) trước
    ws_Z = {s: build_WS(s)['Z'] for s in SCENS}

    m = pyo.ConcreteModel()
    m.J = pyo.Set(initialize=ITEMS)
    m.S = pyo.Set(initialize=SCENS)
    m.x = pyo.Var(m.J, within=pyo.NonNegativeReals)
    m.y = pyo.Var(m.S, m.J, within=pyo.NonNegativeReals)
    m.theta = pyo.Var(within=pyo.NonNegativeReals)   # max regret

    # Minimize max regret
    m.obj = pyo.Objective(expr=m.theta, sense=pyo.minimize)

    # Regret constraint: theta >= Z_WS(s) - [β·x + β^s·y^s]
    def regret_rule(m, s):
        achieved = (sum(BETA[j] * m.x[j] for j in ITEMS)
                  + sum(BETA_S[s, j] * m.y[s, j] for j in ITEMS))
        return m.theta >= ws_Z[s] - achieved
    m.c_regret = pyo.Constraint(m.S, rule=regret_rule)

    m.c1 = pyo.Constraint(expr=sum(m.x[j] for j in ITEMS) <= BUDGET1)
    m.c4 = pyo.Constraint(expr=m.x['H'] >= MIN_H)
    def c5_r(m, j): return m.x[j] >= MIN_ITEM
    m.c5 = pyo.Constraint(m.J, rule=c5_r)
    def c2_r(m, s): return sum(m.y[s, j] for j in ITEMS) <= BUDGET2[s]
    m.c2 = pyo.Constraint(m.S, rule=c2_r)
    def c3_r(m, s): return m.y[s, 'AI'] <= 0.5 * m.x['H']
    m.c3 = pyo.Constraint(m.S, rule=c3_r)

    SolverFactory('highs').solve(m, tee=False)

    x_val = {j: pyo.value(m.x[j]) for j in ITEMS}
    y_val = {(s, j): pyo.value(m.y[s, j]) for s in SCENS for j in ITEMS}
    return {
        'x': x_val, 'y': y_val,
        'max_regret': pyo.value(m.theta),
        'ws_Z': ws_Z,
    }


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Define global constants used in the functions and printing
SCENS = list(SCEN_INFO.keys())
ITEMS = list(BETA.keys())
ITEMS_F = list(BETA.keys()) # Assuming ITEMS_F are just the item keys for now, adjust if needed

print('\n[ Câu 10.5.1 ] Giải SP bằng Pyomo (solver HiGHS)')
print('-' * 65)

res_SP = build_SP()
assert res_SP['feasible'], 'SP không khả thi!'
Z_SP  = res_SP['Z']
x_SP  = res_SP['x']
y_SP  = res_SP['y']

print(f'  ✓ Trạng thái: TỐI ƯU  |  Z* = {Z_SP:,.4f}  (tỷ VND GDP gain)\n')
print('  First-stage decision x_j* (here-and-now):')
for j, jf in zip(ITEMS, ITEMS_F):
    print(f'    {jf:<30}: x[{j}] = {x_SP[j]:>10,.1f} tỷ  ({x_SP[j]/BUDGET1*100:.1f}%)')
print(f'  Tổng first-stage: {sum(x_SP.values()):>10,.1f} / {BUDGET1:,} tỷ')

print('\n  Second-stage recourse y^s_j* (wait-and-see):')
print(f'  {"Hạng mục":<20}' + ''.join(f'  {s:>10}' for s in SCENS))
print('  ' + '-' * 60)
for j in ITEMS:
    row = f'  {ITEMS_F[ITEMS.index(j)]:<20}'
    for s in SCENS:
        row += f'  {y_SP[s,j]:>10,.1f}'
    print(row)
print(f'  {"Tổng y^s":<20}' +
      ''.join(f'  {sum(y_SP[s,j] for j in ITEMS):>10,.1f}' for s in SCENS))
print(f'  {"Budget2^s":<20}' +
      ''.join(f'  {BUDGET2[s]:>10,}' for s in SCENS))

# Kiểm tra ràng buộc C3
print(f'\n  Kiểm tra C3 (y_AI ≤ 0.5·x_H, x_H = {x_SP["H"]:,.0f}):')
for s in SCENS:
    ok = y_SP[s,'AI'] <= 0.5*x_SP['H'] + 1
    print(f'    y[{s},AI] = {y_SP[s,"AI"]:>8,.1f}  ≤  {0.5*x_SP["H"]:>8,.1f}  '+
          f'{"✓" if ok else "✗"}')


[ Câu 10.5.1 ] Giải SP bằng Pyomo (solver HiGHS)
-----------------------------------------------------------------
  ✓ Trạng thái: TỐI ƯU  |  Z* = 92,846.2500  (tỷ VND GDP gain)

  First-stage decision x_j* (here-and-now):
    I                             : x[I] =    3,000.0 tỷ  (4.6%)
    D                             : x[D] =    3,000.0 tỷ  (4.6%)
    AI                            : x[AI] =   44,000.0 tỷ  (67.7%)
    H                             : x[H] =   15,000.0 tỷ  (23.1%)
  Tổng first-stage:   65,000.0 / 65,000 tỷ

  Second-stage recourse y^s_j* (wait-and-see):
  Hạng mục                      s1          s2          s3          s4
  ------------------------------------------------------------
  I                            0.0         0.0         0.0         0.0
  D                        7,500.0     7,500.0         0.0         0.0
  AI                       7,500.0     7,500.0         0.0         0.0
  H                            0.0         0.0    12,000.0     8,000.0
  Tổ

In [ ]:
print('\n[ Câu 10.5.2 ] So sánh SP vs. EV (deterministic)')
print('-' * 65)

# EV: dùng E[β^s_j] thay cho β^s_j theo từng kịch bản
beta_s_EV = {(s, j): sum(P_S[ss] * BETA_S[ss, j] for ss in SCENS)
              for s in SCENS for j in ITEMS}
# EV không biết budget giảm → dùng budget tối đa
budget2_EV = {s: RESERVE for s in SCENS}

res_EV = build_SP(beta_s_override=beta_s_EV, budget2_override=budget2_EV)
x_EV   = res_EV['x']
Z_EV   = res_EV['Z']

# EEV: evaluate x_EV dưới bài toán SP thực
res_EEV = build_SP(fix_x=x_EV)
Z_EEV   = res_EEV['Z']

print(f'  EV (deterministic với E[β^s]):')
for j in ITEMS:
    print(f'    x_EV[{j}] = {x_EV[j]:>10,.1f}  |  x_SP[{j}] = {x_SP[j]:>10,.1f}')
print(f'  EV objective (lạc quan): {Z_EV:>12,.4f}')
print(f'  EEV = E[SP | x=x_EV]:   {Z_EEV:>12,.4f}')
print(f'  SP  = E[SP | x=x_SP*]:  {Z_SP:>12,.4f}')

same_x = all(abs(x_EV[j] - x_SP[j]) < 1 for j in ITEMS)
print(f'\n  x_EV ≈ x_SP? {"Có" if same_x else "Không"} '
      f'(sai số tối đa = {max(abs(x_EV[j]-x_SP[j]) for j in ITEMS):.2f})')

# Practical comparison với naive strategies
X_NAIVE = {
    'AI-heavy\n(bỏ qua rủi ro)':  {'I':3000,'D':3000,'AI':44000,'H':15000},
    'H-conservative\n(né tránh)': {'I':5000,'D':5000,'AI':15000,'H':40000},
    'Cân bằng\n(truyền thống)':   {'I':12000,'D':10000,'AI':20000,'H':23000},
}
print(f'\n  So sánh SP vs. chiến lược naive (Practical Value of SP):')
print(f'  {"Chiến lược":<30} {"Z_eval":>12}  {"Gap vs SP":>12}  {"Gap%":>8}')
print('  ' + '-' * 68)
print(f'  {"SP (stochastic optimal)":<30} {Z_SP:>12,.2f}  {"—":>12}  {"—":>8}')
for name, xn in X_NAIVE.items():
    r = build_SP(fix_x=xn)
    z_n = r['Z']
    gap  = Z_SP - z_n
    gpct = gap / Z_SP * 100
    short_name = name.replace('\n', ' ')
    print(f'  {short_name:<30} {z_n:>12,.2f}  {gap:>+12,.2f}  {gpct:>7.2f}%')



[ Câu 10.5.2 ] So sánh SP vs. EV (deterministic)
-----------------------------------------------------------------
  EV (deterministic với E[β^s]):
    x_EV[I] =    3,000.0  |  x_SP[I] =    3,000.0
    x_EV[D] =    3,000.0  |  x_SP[D] =    3,000.0
    x_EV[AI] =   44,000.0  |  x_SP[AI] =   44,000.0
    x_EV[H] =   15,000.0  |  x_SP[H] =   15,000.0
  EV objective (lạc quan):  93,025.0000
  EEV = E[SP | x=x_EV]:    92,846.2500
  SP  = E[SP | x=x_SP*]:   92,846.2500

  x_EV ≈ x_SP? Có (sai số tối đa = 0.00)

  So sánh SP vs. chiến lược naive (Practical Value of SP):
  Chiến lược                           Z_eval     Gap vs SP      Gap%
  --------------------------------------------------------------------
  SP (stochastic optimal)           92,846.25             —         —
  AI-heavy (bỏ qua rủi ro)          92,846.25         +0.00     0.00%
  H-conservative (né tránh)         85,502.50     +7,343.75     7.91%
  Cân bằng (truyền thống)           87,656.25     +5,190.00     5.59%


In [ ]:
print('\n[ Câu 10.5.3 ] Tính VSS và EVPI')
print('-' * 65)

# WS: Wait-and-See per scenario
WS_results = {}
Z_WS = 0.0
print('  Wait-and-See (thông tin hoàn hảo) từng kịch bản:')
print(f'  {"Kịch bản":<15} {"p":>6} {"Z_WS":>12} {"x_AI":>10} '
      f'{"x_H":>10} {"y_AI":>10} {"y_H":>10}')
print('  ' + '-' * 75)
for s in SCENS:
    ws = build_WS(s)
    WS_results[s] = ws
    Z_WS += P_S[s] * ws['Z']
    print(f'  {SCEN_INFO[s]["label"]:<15} {P_S[s]:>6.2f} '
          f'{ws["Z"]:>12,.2f} {ws["x"]["AI"]:>10,.0f} '
          f'{ws["x"]["H"]:>10,.0f} {ws["y"]["AI"]:>10,.0f} '
          f'{ws["y"]["H"]:>10,.0f}')

VSS  = Z_SP - Z_EEV
EVPI = Z_WS - Z_SP

print(f'\n  Tổng kết VSS và EVPI:')
print(f'  SP   (Stochastic Program)  = {Z_SP:>14,.4f}')
print(f'  EEV  (EV solution in SP)   = {Z_EEV:>14,.4f}')
print(f'  WS   (Wait-and-See)        = {Z_WS:>14,.4f}')
print(f'  VSS  = SP − EEV            = {VSS:>14,.4f}')
print(f'  EVPI = WS − SP             = {EVPI:>14,.4f}')
print(f'\n  Diễn giải:')
if abs(VSS) < 0.01:
    print(f'  → VSS ≈ 0: Trong LP tuyến tính với "relatively complete recourse",')
    print(f'    x_SP* = x_EV* (cùng một nghiệm góc tối ưu). Đây là kết quả đúng')
    print(f'    theo lý thuyết Birge & Louveaux (2011). VSS > 0 xuất hiện khi có')
    print(f'    biến nguyên (MIP) hoặc feasibility regions khác nhau theo scenario.')
if abs(EVPI) < 0.01:
    print(f'  → EVPI ≈ 0: Khi x* không thay đổi theo scenario (same optimal x),')
    print(f'    WS = SP. Thông tin hoàn hảo không tạo ra lợi thế về FIRST-STAGE.')
    print(f'    Tuy nhiên thông tin scenario vẫn rất quan trọng cho SECOND-STAGE:')
    print(f'    y^s1_AI={y_SP["s1","AI"]:,.0f} tỷ (lạc quan) vs '
          f'y^s4_AI={y_SP["s4","AI"]:,.0f} tỷ (khủng hoảng).')

# Tính regret cho từng scenario
print(f'\n  Regret phân tích:')
print(f'  {"Kịch bản":<15} {"Z_WS":>12} {"Z_SP_eval":>12} {"Regret":>12}')
print('  ' + '-' * 53)
for s in SCENS:
    z_sp_s = (sum(BETA[j]*x_SP[j] for j in ITEMS)
            + sum(BETA_S[s,j]*y_SP[s,j] for j in ITEMS))
    regret = WS_results[s]['Z'] - z_sp_s
    print(f'  {SCEN_INFO[s]["label"]:<15} {WS_results[s]["Z"]:>12,.2f} '
          f'{z_sp_s:>12,.2f} {regret:>12,.2f}')


[ Câu 10.5.3 ] Tính VSS và EVPI
-----------------------------------------------------------------
  Wait-and-See (thông tin hoàn hảo) từng kịch bản:
  Kịch bản             p         Z_WS       x_AI        x_H       y_AI        y_H
  ---------------------------------------------------------------------------
  Lạc quan          0.30    97,300.00     44,000     15,000      7,500          0
  Cơ sở             0.45    93,175.00     44,000     15,000      7,500          0
  Bi quan           0.20    87,550.00     44,000     15,000          0     12,000
  Khủng hoảng       0.05    84,350.00     44,000     15,000          0      8,000

  Tổng kết VSS và EVPI:
  SP   (Stochastic Program)  =    92,846.2500
  EEV  (EV solution in SP)   =    92,846.2500
  WS   (Wait-and-See)        =    92,846.2500
  VSS  = SP − EEV            =         0.0000
  EVPI = WS − SP             =         0.0000

  Diễn giải:
  → VSS ≈ 0: Trong LP tuyến tính với "relatively complete recourse",
    x_SP* = x_EV* (cùng 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mticker

print('\n[ Câu 10.5.4 ] Robust Optimization — Minimax Regret')
print('-' * 65)

res_rob = build_robust_minimax()
x_rob   = res_rob['x']
max_reg = res_rob['max_regret']
ws_Z    = res_rob['ws_Z']

# Evaluate x_rob in SP
res_rob_eval = build_SP(fix_x=x_rob)
Z_rob_in_SP  = res_rob_eval['Z']

print(f'  Minimax regret tối thiểu: {max_reg:>12,.4f} tỷ VND')
print(f'\n  So sánh first-stage x_j:')
print(f'  {"Hạng mục":<20} {"x_SP":>12} {"x_Robust":>12} {"Δ":>10}')
print('  ' + '-' * 56)
for j, jf in zip(ITEMS, ITEMS_F):
    delta = x_rob[j] - x_SP[j]
    print(f'  {jf:<20} {x_SP[j]:>12,.1f} {x_rob[j]:>12,.1f} {delta:>+10,.1f}')
print(f'\n  Z trong SP:   SP optimal = {Z_SP:>12,.4f}')
print(f'               Robust      = {Z_rob_in_SP:>12,.4f}')
print(f'  Chi phí robust (SP - Rob) = {Z_SP - Z_rob_in_SP:>12,.4f}  '
      f'({(Z_SP-Z_rob_in_SP)/Z_SP*100:.2f}%)')

print(f'\n  Regret của từng nghiệm theo scenario:')
print(f'  {"Kịch bản":<15} {"Regret(SP)":>12} {"Regret(Rob)":>14}')
print('  ' + '-' * 43)

reg_SP_values  = []
reg_Rob_values = []

for s in SCENS:
    z_sp_s = (sum(BETA[j]*x_SP[j] for j in ITEMS)
            + sum(BETA_S[s,j]*y_SP[s,j] for j in ITEMS))
    y_rob_s = {j: res_rob['y'][s,j] for j in ITEMS}
    z_rob_s = (sum(BETA[j]*x_rob[j] for j in ITEMS)
             + sum(BETA_S[s,j]*y_rob_s[j] for j in ITEMS))
    reg_sp  = ws_Z[s] - z_sp_s
    reg_rob = ws_Z[s] - z_rob_s
    print(f'  {SCEN_INFO[s]["label"]:<15} {reg_sp:>12,.2f} {reg_rob:>14,.2f}')
    reg_SP_values.append(reg_sp)
    reg_Rob_values.append(reg_rob)


[ Câu 10.5.4 ] Robust Optimization — Minimax Regret
-----------------------------------------------------------------
  Minimax regret tối thiểu:       0.0000 tỷ VND

  So sánh first-stage x_j:
  Hạng mục                     x_SP     x_Robust          Δ
  --------------------------------------------------------
  I                         3,000.0      3,000.0       +0.0
  D                         3,000.0      3,000.0       +0.0
  AI                       44,000.0     44,000.0       +0.0
  H                        15,000.0     15,000.0       +0.0

  Z trong SP:   SP optimal =  92,846.2500
               Robust      =  92,846.2500
  Chi phí robust (SP - Rob) =       0.0000  (0.00%)

  Regret của từng nghiệm theo scenario:
  Kịch bản          Regret(SP)    Regret(Rob)
  -------------------------------------------
  Lạc quan                0.00           0.00
  Cơ sở                   0.00           0.00
  Bi quan                 0.00           0.00
  Khủng hoảng             0.00        